In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


Torch: 2.6.0+cu124
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [1]:
import pandas as pd

In [2]:
data_true = pd.read_csv('../data/True.csv')
data_fake = pd.read_csv('../data/Fake.csv')

Noticias reales

In [12]:
data_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


Noticias falsas

In [ ]:
data_fake.head()

In [3]:
#Etiquetar data real y falsa
data_true['label'] = 1
data_fake['label'] = 0

#Eliminar tittle, subject y date
data_true = data_true.drop(['title', 'subject', 'date'], axis=1)
data_fake = data_fake.drop(['title', 'subject', 'date'], axis=1)

Noticas reales

In [5]:
data_true.head()

,text,label
0,WASHINGTON (Reuters) - The head of a conservat...,1
1,WASHINGTON (Reuters) - Transgender people will...,1
2,WASHINGTON (Reuters) - The special counsel inv...,1
3,WASHINGTON (Reuters) - Trump campaign adviser ...,1
4,SEATTLE/WASHINGTON (Reuters) - President Donal...,1


In [4]:
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm
import pandas as pd
import math
import torch

c:\Users\cance\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Configuración
MODEL_NAME = "Helsinki-NLP/opus-mt-en-es"
BATCH_SIZE = 32  # CPU: podemos usar batches grandes
CHUNK_SIZE = 500  # Dividir dataset en chunks para no saturar RAM

# Cargar dataset
data_true["text_es"] = ""

# Cargar modelo y tokenizer
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME)

device = 0  # Solo CPU
model = model.to(device)

# Dividir en chunks
num_chunks = math.ceil(len(data_true) / CHUNK_SIZE)

for c in range(num_chunks):
    start_idx = c * CHUNK_SIZE
    end_idx = min((c+1) * CHUNK_SIZE, len(data_true))
    chunk_texts = data_true["text"].iloc[start_idx:end_idx].tolist()
    
    # Barra de progreso por chunk
    with tqdm(total=len(chunk_texts), desc=f"Traduciendo chunk {c+1}/{num_chunks}", ncols=100) as pbar:
        num_batches = math.ceil(len(chunk_texts) / BATCH_SIZE)
        
        for b in range(num_batches):
            batch_texts = chunk_texts[b*BATCH_SIZE:(b+1)*BATCH_SIZE]
            batch_texts = [txt if isinstance(txt, str) else "" for txt in batch_texts]
            
            # Tokenizar
            encoded = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            
            # Generar traducción
            with torch.no_grad():
                generated_tokens = model.generate(**encoded)
            
            # Decodificar
            translated_texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            
            # Guardar en el dataset
            for i, t in enumerate(translated_texts):
                data_true.loc[start_idx + b*BATCH_SIZE + i, "text_es"] = t
            
            pbar.update(len(batch_texts))

# Guardar resultados finales
data_true.to_csv("../data/dataset_true_traducido.csv", index=False)
print("✔ Traducción completada y guardada en dataset_true_traducido.csv")

c:\Users\NICOLE\Documents\GitHub\1ACC0219-TP-TF-2025-2\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
c:\Users\NICOLE\Documents\GitHub\1ACC0219-TP-TF-2025-2\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\NICOLE\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-es. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to a

✔ Traducción completada y guardada en dataset_true_traducido.csv


In [ ]:
dataset_true_traducido = pd.read_csv('../data/dataset_true_traducido.csv')
dataset_true_traducido.text_es.head()

,text,label,text_es
0,WASHINGTON (Reuters) - The head of a conservat...,1,WASHINGTON (Reuters) - El jefe de una facción ...
1,WASHINGTON (Reuters) - Transgender people will...,1,"El gobierno de EE.UU., el gobierno de EE.UU., ..."
2,WASHINGTON (Reuters) - The special counsel inv...,1,WASHINGTON (Reuters) - La investigación especi...
3,WASHINGTON (Reuters) - Trump campaign adviser ...,1,WASHINGTON (Reuters) - El asesor de la campaña...
4,SEATTLE/WASHINGTON (Reuters) - President Donal...,1,El presidente de EE.UU. llama al Servicio Post...
5,"WEST PALM BEACH, Fla./WASHINGTON (Reuters) - T...",1,La Cámara Blanca dijo el viernes que estaba a ...
6,"WEST PALM BEACH, Fla (Reuters) - President Don...",1,"WEST PALM BEACH, Fla (Reuters) - El presidente..."
7,The following statements were posted to the ve...,1,Las siguientes declaraciones fueron publicadas...
8,The following statements were posted to the ve...,1,Las siguientes declaraciones fueron publicadas...
9,WASHINGTON (Reuters) - Alabama Secretary of St...,1,WASHINGTON (Reuters) - El Secretario de Estado...


Noticias falsas

In [ ]:
data_fake.head()
data_fake.sum()

In [5]:
import torch
import math
from transformers import MarianTokenizer, MarianMTModel
from tqdm import tqdm

# Configuración
MODEL_NAME = "Helsinki-NLP/opus-mt-en-es"
BATCH_SIZE = 32
CHUNK_SIZE = 500

# Detectar device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✔ Mostrar qué dispositivo se está usando
if torch.cuda.is_available():
    print("➡ Usando GPU:", torch.cuda.get_device_name(0))
else:
    print("➡ Usando CPU (no se detectó GPU)")

print("Device interno:", device)

# Cargar dataset
data_fake["text_es"] = ""

# Cargar modelo y tokenizer
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME).to(device)

# Dividir en chunks
num_chunks = math.ceil(len(data_fake) / CHUNK_SIZE)

for c in range(num_chunks):
    start_idx = c * CHUNK_SIZE
    end_idx = min((c + 1) * CHUNK_SIZE, len(data_fake))
    chunk_texts = data_fake["text"].iloc[start_idx:end_idx].tolist()

    # Barra de progreso
    with tqdm(total=len(chunk_texts), desc=f"Traduciendo chunk {c+1}/{num_chunks}", ncols=100) as pbar:
        num_batches = math.ceil(len(chunk_texts) / BATCH_SIZE)

        for b in range(num_batches):
            batch_texts = chunk_texts[b*BATCH_SIZE:(b+1)*BATCH_SIZE]
            batch_texts = [txt if isinstance(txt, str) else "" for txt in batch_texts]

            # Tokenización
            encoded = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            # Generar traducción
            with torch.no_grad():
                generated_tokens = model.generate(**encoded)

            # Decodificar
            translated_texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

            # Guardar en dataset
            for i, t in enumerate(translated_texts):
                data_fake.loc[start_idx + b*BATCH_SIZE + i, "text_es"] = t

            pbar.update(len(batch_texts))

# Guardar dataset traducido
data_fake.to_csv("../data/dataset_fake_traducido.csv", index=False)
print("✔ Traducción completada y guardada.")


➡ Usando CPU (no se detectó GPU)
Device interno: cpu


c:\Users\cance\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Traduciendo chunk 1/47:   0%|                                               | 0/500 [00:11<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
#Combinar datasets
data = pd.concat([data_true, data_fake], ignore_index=True)

# Mezcla aleatoriamente los datos
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

data.to_csv('../data/news_dataset.zip', index=False, compression={'method': 'zip', 'archive_name': 'news_dataset.csv'})